# Cinq stratégies rhétoriques de contre-argument — gabarits déterministes, sans LLM

**Notebook pédagogique CoursIA** · #1961 Phase 5 (assets CoursIA réutilisables) · `Claude Code @ myia-po-2025:2025-Epita-Intelligence-Symbolique`.

Corpus-free : tous les exemples sont synthétiques, domaine-public. Aucun LLM, aucune JVM — la génération est **gabarit** : la version d'origine dépendante du LLM a été remplacée par des gabarits déterministes, et le gabarit statistique **refuse d'inventer des chiffres**.

Ce que couvre ce notebook :

1. les **5 stratégies rhétoriques** (questionnement socratique, reductio ad absurdum, contre-analogie, appel à l'autorité, preuve statistique) et leur **pont prompt** vers un LLM ;
2. la **suggestion de stratégie** — heuristique de contenu d'abord (« statistiques », « tous »…), puis table par type d'argument (déductif/inductif/abductif) ;
3. le **meilleur choix par type de contre-argument** (réfutation directe, contre-exemple, …) ;
4. la **génération réelle** et son point d'honnêteté : le gabarit statistique rend un placeholder explicite plutôt qu'un chiffre fabriqué.

**Garde round-trip** : chaque verdict affiché ici est rejoué contre le moteur source par `tests/unit/coursia/counter_argument_strategies/test_counter_argument_strategies_roundtrip.py` (mêmes exemples, même JSON partagé).


## §1 — Le modèle : cinq stratégies, un pont prompt

Chaque stratégie a un nom, une fonction d'application gabarit, et une **instruction de prompt** (`get_strategy_prompt`) qui décrit au LLM comment s'y prendre — le moteur est bimode : gabarit déterministe sans LLM, instruction dédiée avec LLM.


In [1]:
# Imports + configuration. La génération est déterministe (aucun LLM, aucune JVM).
import json
import logging
import os
import sys

logging.disable(logging.INFO)  # sorties propres malgré la chaîne d'import du dépôt

# Localiser la racine du dépôt (contient argumentation_analysis/) en remontant depuis le CWD.
_cwd = os.getcwd()
while _cwd and not os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    _parent = os.path.dirname(_cwd)
    if _parent == _cwd:
        break
    _cwd = _parent
ROOT = _cwd if os.path.isdir(os.path.join(_cwd, "argumentation_analysis")) else os.getcwd()
sys.path.insert(0, ROOT)

from argumentation_analysis.agents.core.counter_argument.definitions import (
    Argument,
    CounterArgumentType,
    RhetoricalStrategy,
)
from argumentation_analysis.agents.core.counter_argument.strategies import (
    RhetoricalStrategies,
)

EXAMPLES_PATH = os.path.join(
    ROOT, "docs", "coursia_contrib", "counter_argument_strategies_examples.json"
)
with open(EXAMPLES_PATH, encoding="utf-8") as fh:
    examples = json.load(fh)

rs = RhetoricalStrategies()
print(f"{len(rs.strategies)} stratégies rhétoriques :\n")
for strat in rs.strategies:
    print(f"  {strat.name:<22} {strat.value}")
    print(f"     prompt LLM : {rs.get_strategy_prompt(strat)[:72]}…")


5 stratégies rhétoriques :

  SOCRATIC_QUESTIONING   socratic_questioning
     prompt LLM : Use the Socratic method: ask questions that challenge the argument's ass…
  REDUCTIO_AD_ABSURDUM   reductio_ad_absurdum
     prompt LLM : Show that the argument leads to absurd or contradictory consequences.…
  ANALOGICAL_COUNTER     analogical_counter
     prompt LLM : Use a relevant analogy to illustrate the argument's flaws.…
  AUTHORITY_APPEAL       authority_appeal
     prompt LLM : Appeal to recognized authorities or experts to contradict the argument.…
  STATISTICAL_EVIDENCE   statistical_evidence
     prompt LLM : Use statistical data or studies to contradict the argument.…


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2026-09-14 23:21:15 [WARNING] [Services.CryptoService] crypto_service.__init__:46 - Service de chiffrement initialisé sans clé. Le chiffrement est désactivé.


## §2 — Suggestion : le contenu d'abord, le type ensuite

`suggest_strategy(type, contenu)` lit le **contenu** avant tout : « statistiques » ou « données » → preuve statistique ; « tous » ou « chaque » → contre-analogie. Contenu neutre ? Alors la **table par type** décide : déductif → reductio, inductif → autorité, abductif → analogie, et le **défaut** est le questionnement socratique. L'ordre des tests est la sémantique : un contenu généralisant court-circuite le type.


In [2]:
# Les 8 suggestions du JSON partagé passent au moteur réel.
for ex in examples["suggest"]:
    got = rs.suggest_strategy(ex["argument_type"], ex["content"])
    assert got == RhetoricalStrategy[ex["expected"]], (ex, got)
    print(f"{ex['expected']:<22} <- type={ex['argument_type']:<10} « {ex['content'][:48]} »")
    print(f"   {ex['why']}\n")

print("8/8 — chaque contexte rend la stratégie attendue.")


STATISTICAL_EVIDENCE   <- type=deductive  « Les statistiques montrent une baisse de rendemen »
   Le contenu contient « statistiques » — l'heuristique de contenu prime sur le type.

STATISTICAL_EVIDENCE   <- type=deductive  « Les données de l'étude sont claires. »
   « données » déclenche la même branche.

ANALOGICAL_COUNTER     <- type=inductive  « Tous les cas observés convergent. »
   « tous » déclenche l'analogie AVANT la table par type (inductif aurait donné l'autorité).

ANALOGICAL_COUNTER     <- type=deductive  « Chaque équipe réagit de la même façon. »
   « chaque » comme « tous » : généralisation → contre-analogie.

REDUCTIO_AD_ABSURDUM   <- type=deductive  « Une règle stricte s'impose ici. »
   Contenu neutre : la table par type mène — déductif → reductio.

AUTHORITY_APPEAL       <- type=inductive  « Une règle stricte s'impose ici. »
   Inductif → appel à l'autorité.

ANALOGICAL_COUNTER     <- type=abductive  « Une règle stricte s'impose ici. »
   Abductif → contre-analogie.


## §3 — Le meilleur choix par type de contre-argument

`get_best_strategy` part de l'autre bout : le **type de contre-argument** que l'on veut produire. Chaque type admet une courte liste de stratégies, dans un ordre de préférence fixe. Réfutation directe → preuve statistique, puis appel à l'autorité ; contre-exemple → contre-analogie, puis questionnement socratique ; défi de prémisse → questionnement socratique, puis preuve statistique ; reductio → reductio.

L'argument départage ensuite ces candidats : si la stratégie que suggère son propre contenu (`suggest_strategy`, §2) figure dans la liste, elle l'emporte ; sinon, c'est le premier candidat. L'argument de base ci-dessous (déductif, sans « tous » ni « données ») suggère la reductio, qui n'appartient qu'à la liste du type reductio, où elle est déjà en tête : pour lui, le premier candidat de chaque type est donc la recommandation. Un argument qui cite des données, lui, recevrait la preuve statistique face à un défi de prémisse.

In [3]:
# Les 5 recommandations par type de contre-argument, avec assertion.
from argumentation_analysis.agents.core.counter_argument.definitions import Argument

def make_arg(v):
    return Argument(
        content=v["content"], premises=list(v["premises"]),
        conclusion=v["conclusion"], argument_type=v["argument_type"], confidence=0.7,
    )

variants = {k: make_arg(v) for k, v in examples["argument_variants"].items()}

for ex in examples["best_by_type"]:
    got = rs.get_best_strategy(variants["base"], CounterArgumentType[ex["counter_type"]])
    assert got == RhetoricalStrategy[ex["expected"]], (ex, got)
    print(f"{ex['counter_type']:<26} -> {got.name}")

print("\n5/5 — chaque type de contre-argument a sa stratégie de tête.")


DIRECT_REFUTATION          -> STATISTICAL_EVIDENCE
COUNTER_EXAMPLE            -> ANALOGICAL_COUNTER
ALTERNATIVE_EXPLANATION    -> ANALOGICAL_COUNTER
PREMISE_CHALLENGE          -> SOCRATIC_QUESTIONING
REDUCTIO_AD_ABSURDUM       -> REDUCTIO_AD_ABSURDUM

5/5 — chaque type de contre-argument a sa stratégie de tête.


## §4 — Génération réelle, et le gabarit qui refuse de mentir

`apply_strategy` produit le texte du contre-argument. Les gabarits s'adaptent au contenu (une prémisse généralisante « tous » reçoit la question de l'exception ; une conclusion « toujours » enchaîne l'absurde). Le point d'honnêteté : le gabarit **statistique** rend un **placeholder explicite** — il cadre l'objection chiffrée sans inventer de nombre. Un générateur qui fabulerait des données serait plus fluide et moins honnête.


In [4]:
# Les 8 générations du JSON partagé, fragment attendu exigé.
for ex in examples["apply"]:
    text = rs.apply_strategy(
        RhetoricalStrategy[ex["strategy"]],
        variants[ex["variant"]],
        CounterArgumentType[ex["counter_type"]],
    )
    assert ex["expected_fragment"] in text, (ex, text)
    print(f"[{ex['strategy']} / {ex['counter_type']}]")
    print(f"   {text}")
    print(f"   fragment exigé : « {ex['expected_fragment']} » — trouvé\n")

print("8/8 — chaque génération contient son fragment attendu.")
print("\nZoom honnêteté — le gabarit statistique complet :")
print(rs.apply_strategy(
    RhetoricalStrategy.STATISTICAL_EVIDENCE,
    variants["base"],
    CounterArgumentType.DIRECT_REFUTATION,
))


[SOCRATIC_QUESTIONING / PREMISE_CHALLENGE]
   Are you certain there are no exceptions to 'Tous les employés perdent leur concentration'? A single counter-example would invalidate this premise.
   fragment exigé : « no exceptions to » — trouvé

[SOCRATIC_QUESTIONING / PREMISE_CHALLENGE]
   On what basis do you establish that 'Le télétravail isole les collaborateurs'? This premise deserves questioning.
   fragment exigé : « On what basis do you establish » — trouvé

[SOCRATIC_QUESTIONING / DIRECT_REFUTATION]
   How can you reconcile your conclusion 'Le télétravail réduit la productivité' with well-documented cases where the opposite occurred?
   fragment exigé : « reconcile your conclusion » — trouvé

[SOCRATIC_QUESTIONING / COUNTER_EXAMPLE]
   What evidence supports this claim?
   fragment exigé : « What evidence supports this claim? » — trouvé

[REDUCTIO_AD_ABSURDUM / REDUCTIO_AD_ABSURDUM]
   If we accept that Le télétravail réduit toujours la productivité, we should also accept that a

## Ce qu'il faut retenir

- **5 stratégies rhétoriques**, chacune bimode : gabarit déterministe + instruction de prompt LLM ;
- la **suggestion** lit le contenu d'abord (« tous », « statistiques »), la table par type ensuite — l'ordre des tests est la sémantique ;
- le **meilleur choix** part du type de contre-argument visé, et le contenu de l'argument départage les candidats de ce type ;
- la génération **gabarit** produit des textes adaptés au contenu — et le gabarit statistique rend un **placeholder explicite** plutôt qu'un chiffre inventé : l'honnêteté est dans le moteur.

Ce notebook est exécuté — les sorties ci-dessus sont réelles, produites par le moteur du dépôt, pas retouchées.

**Références** : moteur : `argumentation_analysis/agents/core/counter_argument/strategies.py` (adaptation du livrable étudiant `2.3.3-generation-contre-argument`, génération LLM remplacée par gabarits) · asset voisin : `counter_argument_quality.ipynb` (l'évaluateur 5 critères — l'aval de ce générateur) · issue #1961 Phase 5 · garde : `tests/unit/coursia/counter_argument_strategies/test_counter_argument_strategies_roundtrip.py`.